# 08 — Practice Exercises with Automated Checks

    **Companion chapter:** `08-practice-exercises.md`

    ## Learning goals

    - Translate mathematical exercises into executable code.
- Use assertions as immediate feedback.
- Practice attention, masking, positions, normalization, and loss.
- Connect tensor operations with Persian linguistic examples.

    ## How to use this notebook

    Run the cells from top to bottom. Read the comments, change small values, and
    rerun the cell. Every notebook ends with practice prompts that can become
    GitHub issues, exercises, or discussion questions.

In [1]:
import math
import numpy as np
import torch
from torch import nn

np.set_printoptions(precision=3, suppress=True)

# Exercise workflow

For each exercise:

1. Read the prompt.
2. Edit and run the **Your attempt** cell.
3. Compare with the solution cell.
4. Change the input and predict the new result before rerunning.

## Exercise 1 — Softmax

Implement a stable softmax and calculate `softmax([1, 3, 0])`.

In [2]:
# Your attempt
scores = np.array([1.0, 3.0, 0.0])

shifted = scores - scores.max()
your_softmax = np.exp(shifted) / np.exp(shifted).sum()
your_softmax

array([0.114, 0.844, 0.042])

In [3]:
# Solution check
expected = np.array([0.114, 0.844, 0.042])
assert np.allclose(your_softmax, expected, atol=1e-3)
print("Exercise 1 passed.")

Exercise 1 passed.


## Exercise 2 — Weighted context vector

Use `α = [0.2, 0.7, 0.1]` and the three value vectors from the chapter.

In [4]:
# Your attempt
alpha = np.array([0.2, 0.7, 0.1])
values = np.array(
    [
        [1.0, 0.0],
        [0.0, 2.0],
        [1.0, 1.0],
    ]
)

context = alpha @ values
context

array([0.3, 1.5])

In [5]:
assert np.allclose(context, [0.3, 1.5])
print("Exercise 2 passed.")

Exercise 2 passed.


## Exercise 3 — Persian dependency roles

For **علی دیروز کتاب را خواند**, create a dictionary identifying useful tokens
for subject, object, and temporal information.

In [6]:
# Your attempt
useful_context = {
    "subject": ["علی"],
    "object": ["کتاب", "را"],
    "time": ["دیروز"],
}

useful_context

{'subject': ['علی'], 'object': ['کتاب', 'را'], 'time': ['دیروز']}

In [7]:
assert "علی" in useful_context["subject"]
assert "کتاب" in useful_context["object"]
assert "دیروز" in useful_context["time"]
print("Exercise 3 passed.")

Exercise 3 passed.


## Exercise 4 — Causal mask

Construct a `4 × 4` additive mask using `0` for visible positions and `-∞`
for future positions.

In [8]:
# Your attempt
length = 4
mask = np.triu(np.full((length, length), -np.inf), k=1)
mask

array([[  0., -inf, -inf, -inf],
       [  0.,   0., -inf, -inf],
       [  0.,   0.,   0., -inf],
       [  0.,   0.,   0.,   0.]])

In [9]:
assert np.all(mask[np.tril_indices(length)] == 0)
assert np.all(np.isneginf(mask[np.triu_indices(length, k=1)]))
print("Exercise 4 passed.")

Exercise 4 passed.


## Exercise 5 — Learned Q, K, and V projections

In [10]:
x = np.array(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
    ]
)

w_q = np.array([[1.0, 0.0], [0.5, 1.0]])
w_k = np.array([[0.5, 1.0], [1.0, 0.0]])
w_v = np.array([[1.0, 1.0], [0.0, 1.0]])

# Your attempt
q = x @ w_q
k = x @ w_k
v = x @ w_v

scores = q @ k.T / np.sqrt(q.shape[-1])
shifted = scores - scores.max(axis=-1, keepdims=True)
weights = np.exp(shifted) / np.exp(shifted).sum(axis=-1, keepdims=True)
output = weights @ v

print("Weights:\n", weights)
print("Output:\n", output)

Weights:
 [[0.225 0.32  0.456]
 [0.332 0.195 0.473]
 [0.212 0.177 0.611]]
Output:
 [[0.68  1.456]
 [0.805 1.473]
 [0.823 1.611]]


In [11]:
assert q.shape == k.shape == v.shape == (3, 2)
assert weights.shape == (3, 3)
assert output.shape == (3, 2)
assert np.allclose(weights.sum(axis=-1), 1.0)
print("Exercise 5 passed.")

Exercise 5 passed.


## Exercise 6 — One sinusoidal position vector

Calculate the four-dimensional positional encoding for position 2.

In [12]:
# Your attempt
position = 2
d_model = 4

pe = np.zeros(d_model)
for i in range(0, d_model, 2):
    denominator = 10000 ** (i / d_model)
    pe[i] = np.sin(position / denominator)
    pe[i + 1] = np.cos(position / denominator)

pe

array([ 0.909, -0.416,  0.02 ,  1.   ])

In [13]:
expected_pe = np.array([0.909, -0.416, 0.020, 1.000])
assert np.allclose(pe, expected_pe, atol=1e-3)
print("Exercise 6 passed.")

Exercise 6 passed.


## Exercise 7 — Layer normalization by hand

In [14]:
# Your attempt
z = np.array([1.5, 1.5, 4.0])
normalized = (z - z.mean()) / np.sqrt(z.var() + 1e-5)
normalized

array([-0.707, -0.707,  1.414])

In [15]:
expected_ln = np.array([-0.707, -0.707, 1.414])
assert np.allclose(normalized, expected_ln, atol=1e-3)
print("Exercise 7 passed.")

Exercise 7 passed.


## Exercise 8 — Cross-entropy

In [16]:
# Your attempt
correct_token_probabilities = np.array([0.7, 0.01])
nll = -np.log(correct_token_probabilities)
nll

array([0.357, 4.605])

In [17]:
assert np.allclose(nll, [0.357, 4.605], atol=1e-3)
assert nll[1] > nll[0]
print("Exercise 8 passed.")

Exercise 8 passed.


## Exercise 9 — Shape reasoning

Use `batch=2`, `heads=4`, `sequence=10`, and `head_dim=8`.
Determine Q/K/V, score, and concatenated output shapes.

In [18]:
batch, heads, sequence, head_dim = 2, 4, 10, 8

qkv_shape = (batch, heads, sequence, head_dim)
score_shape = (batch, heads, sequence, sequence)
concatenated_shape = (batch, sequence, heads * head_dim)

print("Q/K/V:", qkv_shape)
print("Scores:", score_shape)
print("Concatenated:", concatenated_shape)

Q/K/V: (2, 4, 10, 8)
Scores: (2, 4, 10, 10)
Concatenated: (2, 10, 32)


In [19]:
assert qkv_shape == (2, 4, 10, 8)
assert score_shape == (2, 4, 10, 10)
assert concatenated_shape == (2, 10, 32)
print("Exercise 9 passed.")

Exercise 9 passed.


## Exercise 10 — Open-ended repository task

Add one new exercise based on Persian morphology or word order. Include:

- a short prompt;
- starter code;
- an automated assertion where possible;
- a short explanation of what the result means linguistically.